In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split

In [ ]:
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

In [ ]:
stopwords_set=set(stopwords.words('english'))

def clean_text(text):
    text=str(text).lower()
    text=re.sub(r'[^0-9a-zA-Z]',' ',text)
    text=re.sub(r'\s+',' ',text)
    text=' '.join(word for word in text.split() if word not in stopwords_set)
    return text

In [ ]:
# Available dataset choices
DATASET_OPTIONS = ['SMS', 'Email', 'Both']

def load_dataset(dataset_choice='SMS'):
    """Load and preprocess the chosen dataset.
    dataset_choice: 'SMS', 'Email', or 'Both'
    Returns: X_train, X_test, y_train, y_test
    """
    frames = []

    if dataset_choice in ('SMS', 'Both'):
        sms = pd.read_csv('data/spam.csv', encoding='latin1')
        sms = sms[['v1', 'v2']]
        sms.rename(columns={'v1': 'label', 'v2': 'message'}, inplace=True)
        sms['label'] = sms['label'].map({'ham': 0, 'spam': 1})
        frames.append(sms)

    if dataset_choice in ('Email', 'Both'):
        email = pd.read_csv('data/emailSpam.csv', encoding='latin1')
        email = email[['Body', 'Label']]
        email.rename(columns={'Body': 'message', 'Label': 'label'}, inplace=True)
        frames.append(email)

    df = pd.concat(frames, ignore_index=True)
    df.dropna(subset=['message'], inplace=True)
    df['cleaned_text'] = df['message'].apply(clean_text)

    X = df['cleaned_text']
    y = df['label']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    return X_train, X_test, y_train, y_test, df

In [ ]:
def display_distribution(df):
    plt.figure(figsize=(6, 4))
    sns.countplot(data=df, x='label')
    plt.title('Distribution of Spam vs Ham Messages')
    plt.xlabel('Label')
    plt.ylabel('Count')
    plt.show()

def display_dataset(df):
    return df

In [ ]:
# Default load: SMS (backwards compatible with existing notebooks)
X_train, X_test, y_train, y_test, df = load_dataset('SMS')